# nb_08_search_examples — how to query the RAG index

A runnable reference for querying the `docs-rag` Azure AI Search index this pipeline builds. Each
section is a self-contained example: **keyword**, **filtered**, **security-trimmed**, **vector**,
**hybrid**, and **semantic** search. This is the query side an application (or an agent's
retrieval tool) would use.

## Inputs
- `config` keys: `search_endpoint`, `search_index_name`, `kv_name`, `search_key_secret`,
  `aoai_endpoint`, `aoai_embedding_deployment`.
- Key Vault Search key (query/admin) + an Entra token for Azure OpenAI (vector/hybrid embed the
  query text). Same auth as nb_03.

## How to run
Attach the lakehouse and Run All, or edit `QUERY` / `USER_GROUPS` in the parameters cell and run
individual example cells. Read-only — issues Search queries and one AOAI embedding call.

> **Security trimming is the important pattern:** always pass the caller's Entra group IDs into
> the `allowed_groups/any(...)` filter so users only ever see chunks they're entitled to.


## Parameters


In [ ]:
QUERY = 'quarterly revenue growth and operating margins'
# Entra group object IDs the calling user belongs to (drives security trimming).
USER_GROUPS = ['11111111-1111-1111-1111-111111111111']
TOP = 5


## Config + auth
Search admin/query key from Key Vault; an Entra `cognitiveservices.azure.com` token for the AOAI
embedding call used by vector/hybrid search.


In [ ]:
import requests, notebookutils, json
cfg = {r['key']: r['value'] for r in spark.table('config').collect()}
SEARCH_ENDPOINT = cfg['search_endpoint'].rstrip('/')
INDEX_NAME = cfg['search_index_name']
SEARCH_API = '2024-07-01'
VAULT_URL = f"https://{cfg['kv_name']}.vault.azure.net/"
SEARCH_KEY = notebookutils.credentials.getSecret(VAULT_URL, cfg['search_key_secret'])
HDRS = {'api-key': SEARCH_KEY, 'Content-Type': 'application/json'}
AOAI_ENDPOINT = cfg['aoai_endpoint'].rstrip('/')
EMB_DEPLOY = cfg['aoai_embedding_deployment']
AOAI_API = '2024-06-01'
SEARCH_URL = f'{SEARCH_ENDPOINT}/indexes/{INDEX_NAME}/docs/search?api-version={SEARCH_API}'
def cog_token():
    return notebookutils.credentials.getToken('https://cognitiveservices.azure.com')
print('index:', INDEX_NAME)


## Helpers
`run_search` POSTs a query body and pretty-prints the top hits; `trimming_filter` builds the
security filter from a list of group IDs; `embed_query` turns the query text into a vector.


In [ ]:
def trimming_filter(groups):
    quoted = ', '.join("'" + g + "'" for g in groups)
    return f'allowed_groups/any(g: search.in(g, {quoted}))' if groups else None

def embed_query(text):
    url = f'{AOAI_ENDPOINT}/openai/deployments/{EMB_DEPLOY}/embeddings?api-version={AOAI_API}'
    h = {'Authorization': f'Bearer {cog_token()}', 'Content-Type': 'application/json'}
    r = requests.post(url, headers=h, json={'input': [text]}, timeout=(10, 60))
    r.raise_for_status()
    return r.json()['data'][0]['embedding']

def run_search(body, label):
    r = requests.post(SEARCH_URL, headers=HDRS, json=body, timeout=(10, 60))
    r.raise_for_status()
    hits = r.json().get('value', [])
    print(f'--- {label}: {len(hits)} hits ---')
    for d in hits:
        score = d.get('@search.rerankerScore', d.get('@search.score'))
        name = d.get('file_name'); page = d.get('page_number')
        snippet = (d.get('content') or '').replace('\n', ' ')[:110]
        print(f'  {score:>6.3f}  {name} p{page}  {snippet}')
    return hits


## 1. Keyword search
Plain BM25 keyword search over `content`, restricted to the caller's groups. `select` limits the
returned fields (never return `content_vector` — it's large).


In [ ]:
body = {
    'search': QUERY,
    'filter': trimming_filter(USER_GROUPS),
    'select': 'file_name,file_path,page_number,content',
    'top': TOP,
}
_ = run_search(body, 'keyword (trimmed)')


## 2. Filtered search (structured fields)
Combine the security filter with structured predicates — here: PDFs only, page 1. Filters use
OData syntax and combine with `and`.


In [ ]:
filters = [trimming_filter(USER_GROUPS), "file_extension eq 'pdf'", 'page_number eq 1']
body = {
    'search': QUERY,
    'filter': ' and '.join(f for f in filters if f),
    'select': 'file_name,page_number,file_extension,content',
    'top': TOP,
}
_ = run_search(body, 'filtered (pdf, page 1, trimmed)')


## 3. Security trimming — same query, different users
The heart of the ACL model: identical query text, different `USER_GROUPS`, different results. A
group with no matching docs gets nothing. Never send a query without this filter in production.


In [ ]:
for demo_groups in [USER_GROUPS,
                    ['22222222-2222-2222-2222-222222222222'],
                    ['99999999-9999-9999-9999-999999999999']]:
    body = {'search': QUERY, 'filter': trimming_filter(demo_groups),
            'select': 'file_name,file_path,page_number', 'top': TOP}
    hits = run_search(body, f'groups={demo_groups}')
    print()


## 4. Vector search (semantic similarity)
Embed the query with the same Azure OpenAI model used at ingest, then do a nearest-neighbour
search over `content_vector`. The security filter still applies (trimming works on vector
queries too).


In [ ]:
vec = embed_query(QUERY)
body = {
    'count': True,
    'select': 'file_name,page_number,content',
    'filter': trimming_filter(USER_GROUPS),
    'vectorQueries': [{'kind': 'vector', 'vector': vec,
                       'fields': 'content_vector', 'k': TOP}],
}
_ = run_search(body, 'vector (trimmed)')


## 5. Hybrid search (keyword + vector)
Send both a `search` term and a `vectorQueries` entry; Search fuses the two rankings (RRF). This
is usually the best default for RAG — keyword precision plus vector recall.


In [ ]:
body = {
    'search': QUERY,
    'select': 'file_name,page_number,content',
    'filter': trimming_filter(USER_GROUPS),
    'vectorQueries': [{'kind': 'vector', 'vector': vec,
                       'fields': 'content_vector', 'k': TOP}],
    'top': TOP,
}
_ = run_search(body, 'hybrid (keyword + vector, trimmed)')


## 6. Semantic ranking (optional)
If the index has a semantic configuration, `queryType=semantic` reranks the top results with a
language model and can return captions/answers. This **fails gracefully** if no semantic config
is defined on the index — it's an optional enhancement, not required by the pipeline.


In [ ]:
body = {
    'search': QUERY,
    'queryType': 'semantic',
    'semanticConfiguration': 'default',
    'filter': trimming_filter(USER_GROUPS),
    'select': 'file_name,page_number,content',
    'top': TOP,
}
try:
    _ = run_search(body, 'semantic')
except requests.HTTPError as e:
    print('semantic search not available (no semantic config on index):', e.response.status_code)


## 7. Folder-scoped search + folder facets (Sprint 11 metadata)
The `folder_path` metadata field (parent folder of the source key) supports folder-scoped
retrieval and faceting. `file_size` / `last_modified` are also filterable/sortable. Facets return
the document distribution across folders (useful for a source browser).


In [ ]:
# a) restrict retrieval to a folder subtree
folder = 'testset/finance/reports'
body = {
    'search': QUERY,
    'filter': ' and '.join(f for f in [trimming_filter(USER_GROUPS),
                                       f"folder_path eq '{folder}'"] if f),
    'select': 'file_name,folder_path,page_number,file_size,last_modified,content',
    'top': TOP,
}
_ = run_search(body, f'folder-scoped ({folder}, trimmed)')

# b) folder facets — document counts per folder (no security filter = admin/reporting view)
facet_body = {'search': '*', 'facets': ['folder_path,count:50'], 'top': 0}
fr = requests.post(SEARCH_URL, headers=HDRS, json=facet_body, timeout=(10, 60))
fr.raise_for_status()
print('--- folder_path facets ---')
for f in fr.json().get('@search.facets', {}).get('folder_path', []):
    print(f"  {f['count']:>5}  {f['value']}")


## Notes
- Always pass `allowed_groups/any(g: search.in(g, ...))` with the caller's real Entra group IDs —
  resolve those from the signed-in user (e.g. Microsoft Graph) at query time.
- `page_number` lets you cite the exact source page; `file_path` is the OneLake/S3 location.
- Keep `select` tight (never return `content_vector`).
- `2024-07-01` is the query API version used throughout this solution.
